# **Field Generation** (data from Google Earth Engine)


[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1SIicxds2BeBCZ-JwboAEgO-dSES4xPjq?usp=sharing)

**Author**: Ivan Zvonkov (ivan.zvonkov@gmail.com)

**Last Updated**: Jan 19, 2026

**Description**: The notebook runs the Fields of the World inference pipeline.

**Prerequisites**:
A Google Cloud Project with Earth Engine enabled and a GCS bucket for storing inference artifacts.


## 1. Run setup

In [ ]:
# Environment configuration
# ------------------------------------------------------------------------------
GCLOUD_PROJECT = "name-of-your-project"
BUCKET = "name-of-your-bucket"

# Environment setup
# ------------------------------------------------------------------------------
from contextlib import suppress
from google.cloud import storage
from IPython.display import display_html
from pathlib import Path
from tqdm.notebook import tqdm
import ee
import google
import geopandas as gpd
import pandas as pd
import subprocess
import time

ee.Authenticate()
!earthengine set_project {GCLOUD_PROJECT} # Required for ee CLI

SCOPES = ["https://www.googleapis.com/auth/cloud-platform", "https://www.googleapis.com/auth/earthengine"]
CREDENTIALS, _ = google.auth.default(default_scopes=SCOPES)
ee.Initialize(CREDENTIALS, project=GCLOUD_PROJECT, opt_url='https://earthengine-highvolume.googleapis.com')

bucket = storage.Client().bucket(BUCKET)
tiles_df = pd.read_csv("gs://name-of-your-bucket/S2_Tiles_EcoRegions_GAUL.csv")

def sh(description, cmd):
  print(f"\t{description} ...", end="")
  start = time.perf_counter()
  subprocess.run(cmd, shell=True, check=True)
  duration = time.perf_counter() - start
  print(f"\t{duration:.2f}s\t ✓")

stem = lambda x: Path(x.name).stem
blob_count = lambda x: str(len(list(bucket.list_blobs(prefix=x))))
blob_exists = lambda x: "✓" if bucket.blob(x).exists() else "x"
asset_count = lambda x: str(len(ee.data.listAssets(x)['assets']))

def asset_exists(x):
  try:
    ee.data.getAsset(x)
    return "✓"
  except:
    return "x"

In [ ]:
# Run configuration
# ------------------------------------------------------------------------------
RUN = "Brazil_Minas_Gerais_v20260115"
S2_TILES = set(tiles_df[(tiles_df["ADM0_NAME"] == "Brazil") & (tiles_df["ADM1_NAME"] == "Minas Gerais")]["name"].tolist())
CKPT_MODELS = ["countries_and_biomes_20250926", "TkT_TBL1sol_06172025"]
DEFAULT_MODELS = [] #["FTW_PRUE_EFNET_B7"]
MODELS = CKPT_MODELS + DEFAULT_MODELS

ORDERED_S2_TILES = list(S2_TILES)
ORDERED_S2_TILES.sort()
print()
for t in ORDERED_S2_TILES:
  print(t)

In [ ]:
def create_ee_folder(folder):
  try:
    ee.data.createFolder(folder)
  except ee.EEException:
    print(f"GEE folder: {folder} already exists.")

for MODEL in MODELS:
  create_ee_folder(f"projects/{GCLOUD_PROJECT}/assets/ftw-tifs")
  create_ee_folder(f"projects/{GCLOUD_PROJECT}/assets/ftw-final")
  create_ee_folder(f"projects/{GCLOUD_PROJECT}/assets/ftw-tifs/{MODEL}")
  create_ee_folder(f"projects/{GCLOUD_PROJECT}/assets/ftw-tifs/{MODEL}/{RUN}")
  create_ee_folder(f"projects/{GCLOUD_PROJECT}/assets/ftw-final/{MODEL}")

In [ ]:
# Run status
# ------------------------------------------------------------------------------
print(blob_count(f'input-tifs/{RUN}') + f"/{len(S2_TILES)} S2 tiles available for inference.")
for MODEL in MODELS:
  print(f"\n{MODEL}")
  print(blob_count(f'inference-tifs/{MODEL}/{RUN}') + f"/{len(S2_TILES)} GCS inference raster tiles")
  print(asset_count(f'projects/{GCLOUD_PROJECT}/assets/ftw-tifs/{MODEL}/{RUN}') + f"/{len(S2_TILES)} GEE inference raster tiles")
  print(blob_count(f'masked-inference-tifs/{MODEL}/{RUN}')+     " GCS masked postprocessed raster")
  print(blob_exists(f'postprocessed-parquet/{MODEL}/{RUN}.parquet')+ " GCS merged postprocessed parquet")
  print(blob_exists(f'postprocessed-shapefile/{MODEL}/{RUN}.zip')+   " GCS merged postprocessed shapefile")
  print(asset_exists(f'projects/{GCLOUD_PROJECT}/assets/ftw-final/{MODEL}/{RUN}') +   " GEE asset")

## 2. Export input data

In [ ]:
# Planting and harvest data configuration
# ------------------------------------------------------------------------------
PLANTING_START =  '2023-10-01' # '2023-07-01'
PLANTING_END =    '2024-01-31' # '2023-11-30'
HARVEST_START =   '2024-02-01' # '2024-01-01'
HARVEST_END =     '2024-05-31'

# Sentinel-2 planting and harvest median tiles export to Cloud Storage bucket
# ------------------------------------------------------------------------------
def maskS2Clouds(image):
  cloudProb = image.select('MSK_CLDPRB'); # Cloud probability
  scl = image.select('SCL'); # Scene classification
  # mask cloud shadows (3), medium probability clouds (8), high probability clouds (9), cirrus (10), snow/ice (11)
  cloudMask = cloudProb.lt(30).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11));
  return image.updateMask(cloudMask)

ACTIVE_TASKS = [t for t in ee.data.listOperations() if t["metadata"]["state"] in ["RUNNING", "PENDING"]]
ACTIVE_TILES = {t["metadata"]["description"] for t in ACTIVE_TASKS }
INPUT_TIFS = {stem(b) for b in bucket.list_blobs(prefix=f'input-tifs/{RUN}')}
TILES_READY_FOR_EXPORT = S2_TILES - (INPUT_TIFS | ACTIVE_TILES)

if len(TILES_READY_FOR_EXPORT) > 0:
  pbar = tqdm(TILES_READY_FOR_EXPORT)
  S2TileGeometries = ee.FeatureCollection('users/wiell/SepalResources/sentinel2SceneAreas')
  S2_SR = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  for TILE in pbar:
    pbar.set_description(TILE)
    S2_images = S2_SR.filter(ee.Filter.eq("MGRS_TILE", TILE))
    plantingImages = S2_images.filterDate(PLANTING_START, PLANTING_END).map(maskS2Clouds).select(['B4','B3','B2','B8'])
    harvestImages = S2_images.filterDate(HARVEST_START, HARVEST_END).map(maskS2Clouds).select(['B4','B3','B2','B8'])

    eeProj = plantingImages.first().select('B4').projection()
    proj = eeProj.getInfo()

    plantingMedian = plantingImages.median().rename(['B4wA', 'B3wA', 'B2wA', 'B8wA'])
    harvestMedian = harvestImages.median().rename(['B4wB', 'B3wB', 'B2wB', 'B8wB'])
    ftwInput = plantingMedian.addBands(harvestMedian).toUint16()

    ee.batch.Export.image.toCloudStorage(
      image=ftwInput.reproject(eeProj).clip(S2TileGeometries.filter(f"name == '{TILE}'").geometry()),
      description=TILE,
      bucket=BUCKET,
      fileFormat='GeoTIFF',
      fileNamePrefix=f"input-tifs/{RUN}/{TILE}",
      maxPixels=19885391896,
      crs=proj["crs"],
      crsTransform=proj["transform"],
    ).start()

print(f"{len(ACTIVE_TILES)}/{len(S2_TILES)} active Google Earth Engine exports: https://code.earthengine.google.com/tasks")
print(f"{len(INPUT_TIFS)}/{len(S2_TILES)} tiles ready for inference.")


## 3. Run inference (requires GPU)

In [ ]:
!pip install "ftw-tools==2.0.0b4" ultralytics > dependencies.txt

In [ ]:
# Inference setup
#!pip install ftw-tools "torchgeo<0.7" > dependencies.txt
for MODEL in CKPT_MODELS:
  !gsutil cp -n gs://ftw-bucket/models/{MODEL}.ckpt {MODEL}.ckpt

In [ ]:
# Inference run (this cell is intended to be run several times as data becomes available)
INPUT_TIFS = {stem(b) for b in bucket.list_blobs(prefix=f'input-tifs/{RUN}')}
FIELD_TIFS = {MODEL: {stem(b) for b in bucket.list_blobs(prefix=f'inference-tifs/{MODEL}/{RUN}')} for MODEL in MODELS}
for TILE in tqdm(INPUT_TIFS):
  for MODEL in MODELS:
    if TILE not in FIELD_TIFS[MODEL]:
      print(f'\n{TILE} ({MODEL})')
      if not Path(f"{TILE}.tif").exists():
        sh("Downloading S2 data          ", f"gcloud storage cp -n gs://{BUCKET}/input-tifs/{RUN}/{TILE}.tif {TILE}.tif")
      MRT = f"{MODEL}/{RUN}/{TILE}"
      M_ARG = f"{MODEL}.ckpt" if MODEL in CKPT_MODELS else MODEL
      sh("Running inference            ", f"ftw inference run /content/{TILE}.tif -f -o {MODEL}_{TILE}_fields.tif --gpu 0 -m {M_ARG}")
      sh("Uploading fields tif         ", f"gcloud storage cp -n {MODEL}_{TILE}_fields.tif gs://{BUCKET}/inference-tifs/{MRT}.tif")
      sh("Starting GEE raster upload   ", f"earthengine upload image gs://{BUCKET}/inference-tifs/{MRT}.tif --asset_id projects/{GCLOUD_PROJECT}/assets/ftw-tifs/{MRT}")
      subprocess.run(f"rm {MODEL}_{TILE}_fields.tif", shell=True, check=True)
  Path(f"{TILE}.tif").unlink(missing_ok=True)


## 4. Masking using MapBiomas

In [ ]:
prefix = "projects/mapbiomas-public/assets"
MapBiomasImageCollection = ee.ImageCollection.fromImages([
  ee.Image(f'{prefix}/brazil/lulc/collection10/mapbiomas_brazil_collection10_coverage_v2').select('classification_2024').rename("classification"),
  ee.Image(f'{prefix}/argentina/collection1/mapbiomas_argentina_collection1_integration_v1').select('classification_2022').rename("classification"),
  ee.Image(f'{prefix}/chaco/lulc/collection5/mapbiomas_chaco_collection5_integration_v2').select('classification_2023').rename("classification"),
  ee.Image(f'{prefix}/paraguay/collection1/mapbiomas_paraguay_collection1_integration_v1').select('classification_2022').rename("classification"),
  ee.Image(f'{prefix}/bolivia/lulc/collection3/mapbiomas_bolivia_collection3_integration_v1').select('classification_2023').rename("classification")
])

MapBiomas = MapBiomasImageCollection.mosaic()

fieldClasses = [14, 15, 18, 19, 39, 20, 40, 62, 41, 36, 46, 47, 35, 48, 9, 21, 57, 58]
# fieldClasses.append(12) # Whether to include grassland

fieldMask = MapBiomas.eq(fieldClasses).reduce('sum')

In [ ]:
maskedURLs = []
mosaickedURLs = []
palette = ["black", "lightgreen", "lightgray"]
def visURL(img, geom):
  return img.visualize(min=0, max=2, palette=palette).getThumbUrl({"region": geom, "dimensions": 600})

ACTIVE_TASKS = [t for t in ee.data.listOperations() if t["metadata"]["state"] in ["RUNNING", "PENDING"]]
ACTIVE_TASK_DESCRIPTIONS = {t["metadata"]["description"] for t in ACTIVE_TASKS }

for MODEL in MODELS:

  assets = ee.data.listAssets(f"projects/{GCLOUD_PROJECT}/assets/ftw-tifs/{MODEL}/{RUN}")["assets"]
  images = [ee.Image(asset["id"]) for asset in assets]

  fieldRasterCollection = ee.ImageCollection(images)
  geom = fieldRasterCollection.geometry().dissolve()
  mosaickedFields = fieldRasterCollection.mosaic()
  maskedFields = mosaickedFields.updateMask(fieldMask)

  palette = ["black", "lightgreen", "lightgray"]
  maskedURLs.append(visURL(maskedFields, geom))
  mosaickedURLs.append(visURL(mosaickedFields, geom))

  if f"{MODEL}_{RUN}" in ACTIVE_TASK_DESCRIPTIONS:
    print(f"{MODEL}: GEE task already launched.")
    continue
  elif len(images) != len(S2_TILES):
    print(f"{MODEL}: found {len(images)} field rasters on EarthEngine but expecting {len(S2_TILES)}")
    continue
  else:
    print(f"{MODEL}: found {len(images)} field rasters for each S2 tile, proceeding to export.")

  # Create GEE export task
  proj = images[0].projection().getInfo()
  ee.batch.Export.image.toCloudStorage(
    image=maskedFields,
    description=f"{MODEL}_{RUN}",
    bucket=BUCKET,
    fileNamePrefix=f"masked-inference-tifs/{MODEL}/{RUN}/",
    region=geom,
    scale=10,
    crs=proj["crs"],
    crsTransform=proj["transform"],
    maxPixels=1e13,
    skipEmptyTiles=True,
    fileFormat="GeoTIFF"
  ).start()
print("View masked-inference exports: https://code.earthengine.google.com/tasks")

In [ ]:
legend = f"""
<div>
  <span style="color:{palette[1]};">■</span> Interior pixels
<span style="color:{palette[2]};">■</span> Border pixels
</div>
"""

def maskedAndMosaicked(MODEL, mosaickedURL, maskedURL):
  return f"""
  <h2>{MODEL}</h2>
<div style="display:flex; gap:40px; text-align:center">
  <div><h3>Mosaicked Fields ({len(images)} Tiles)</h3><img src="{mosaickedURL}"/>{legend}</div>
  <div><h3>Masked Fields</h3><img src="{maskedURL}"/>{legend}</div>
</div>"""

html = ""
for MODEL, maskedURL, mosaickedURL in zip(MODELS, maskedURLs, mosaickedURLs):
  html += maskedAndMosaicked(MODEL, mosaickedURL, maskedURL)
display_html(html, raw=True)

## 5. Polygonizing (requires external VM)

In [ ]:
SHP_FILE_STR = "import geopandas as gpd; gpd.read_parquet('merged.parquet').rename(columns={'determination_method': 'det_method'}).to_file('fields', driver='ESRI Shapefile')"

print(f"Polygonize in VM with lot's of memory using following script:")
print(f"""{'-'*100}
sudo apt-get install python3-gdal gdal-bin python3-pip python3.11-venv zip -y
python3 -m venv venv && source venv/bin/activate && pip install ftw-tools
source venv/bin/activate""")

for MODEL in MODELS:
  masked_inference_tifs = [b.name for b in bucket.list_blobs(prefix=f"masked-inference-tifs/{MODEL}/{RUN}/")]
  if len(masked_inference_tifs) == 0:
    print("No masked inference tifs found yet. EarthEngine is probably still exporting.")
  else:
    print(f"""
gcloud storage cp -r -n gs://{BUCKET}/masked-inference-tifs/{MODEL}/{RUN}/ .
gdal_merge.py -co COMPRESS=LZW -o merged.tif {RUN}/*.tif
ftw inference polygonize --min_size 5000 --close_interiors -f merged.tif
gcloud storage cp merged.parquet gs://{BUCKET}/postprocessed-parquet/{MODEL}/{RUN}.parquet

python3 -c "{SHP_FILE_STR}"
zip -r {RUN}.zip fields
gcloud storage cp {RUN}.zip gs://{BUCKET}/postprocessed-shapefile/{MODEL}/{RUN}.zip
rm -r fields {RUN} *.parquet *.zip  merged.tif
  """)

In [ ]:
print("Once merging is complete run the following to ingest each shapefiles into GEE:")
for MODEL in MODELS:
 print(f"earthengine upload table gs://{BUCKET}/postprocessed-shapefile/{MODEL}/{RUN}.zip --asset_id projects/{GCLOUD_PROJECT}/assets/ftw-final/{MODEL}/{RUN}")

Once the shapefile is uploaded to Earth Engine, quality assessment can be conducted on a tile basis using this [script](https://code.earthengine.google.com/1db09824ec22bda9539712c18fd70f40).